In [ ]:
import numpy as np
import pandas as pd
import matplotlib as plt

import os
import sys

sys.path.insert(0, r"C:\ProjectsDannyDavis\VolatilityPredictionProject\utils")

import API_Keys
from File_Paths import raw_data_filepath
import OpenBBConvenienceFunctions as obb_cf
import PlottingFunctions as plot_func
from file_reader import read_file

from statsmodels.tsa.stattools import adfuller, kpss

In [2]:
SAP = read_file(raw_data_filepath, r"SP500_price_data.parquet")

In [3]:
SAP

,open,high,low,close,volume,simple_return,log_return
date,,,,,,,
2010-01-05,1132.660034,1136.630005,1129.660034,1136.520020,2491020000,0.003116,0.003111
2010-01-06,1135.709961,1139.189941,1133.949951,1137.140015,4972660000,0.000546,0.000545
2010-01-07,1136.270020,1142.459961,1131.319946,1141.689941,5270680000,0.004001,0.003993
2010-01-08,1140.520020,1145.390015,1136.219971,1144.979980,4389590000,0.002882,0.002878
2010-01-11,1145.959961,1149.739990,1142.020020,1146.979980,4255780000,0.001747,0.001745
...,...,...,...,...,...,...,...
2025-12-24,6904.910156,6937.319824,6904.910156,6932.049805,1798270000,0.003221,0.003216
2025-12-26,6936.020020,6945.770020,6921.600098,6929.939941,2586550000,-0.000304,-0.000304
2025-12-29,6903.600098,6920.209961,6888.759766,6905.740234,3541750000,-0.003492,-0.003498


In [4]:
var_df = pd.DataFrame(SAP['simple_return']**2)

In [5]:
var_df

,simple_return
date,
2010-01-05,9.707435e-06
2010-01-06,2.975928e-07
2010-01-07,1.600962e-05
2010-01-08,8.304352e-06
2010-01-11,3.051155e-06
...,...
2025-12-24,1.037795e-05
2025-12-26,9.263717e-08
2025-12-29,1.219442e-05


### ADF-test
Full ADF Regression: **Δx_t = α + β·t + δ·x_t−1 + Σi=1p γi·Δxt−i + εt**

```H_0 = Unit root (random process)```

```H_1 = No unit root (stationary)```

In [49]:
adf_result = adfuller(x=var_df, maxlag=25, regression='n', autolag='BIC', regresults=True)
adf_res_store = adf_result[-1]

In [50]:
adf_res_store.adfstat # Augmented Dickey-Fuller Stat

np.float64(-11.180968393414519)

In [51]:
adf_result[1] # P-value -- shows statistical significance

np.float64(3.0386878035291215e-20)

In [54]:
# Number of lags = whichever model minimizes the BIC

print(f"BIC with 1 lag: {adf_res_store.autolag_results[1].bic.round(2)}")
print(f"BIC with 7 lags: {adf_res_store.autolag_results[8].bic.round(2)}")
print(f"BIC with 25 lags: {adf_res_store.autolag_results[26].bic.round(2)}")

BIC with 1 lag: -50847.77
BIC with 7 lags: -51471.33
BIC with 25 lags: -51383.09


In [55]:
print(adf_res_store.resols.summary()) # Full OLS Summary

                                 OLS Regression Results                                
Dep. Variable:                      y   R-squared (uncentered):                   0.378
Model:                            OLS   Adj. R-squared (uncentered):              0.377
Method:                 Least Squares   F-statistic:                              304.7
Date:                Sat, 09 May 2026   Prob (F-statistic):                        0.00
Time:                        18:02:29   Log-Likelihood:                          25890.
No. Observations:                4015   AIC:                                 -5.176e+04
Df Residuals:                    4007   BIC:                                 -5.171e+04
Df Model:                           8                                                  
Covariance Type:            nonrobust                                                  
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------

### KPSS test

Structural Decomposition: **xt = β·t + rt + εt**
                            
**rt = rt_−1 + ut, ut ~ iid(0, σ²u)**

    H_0: σ²_u = 0 (stationary)

    H_1: σ²_u > 0 (unit root)

In [6]:
kpss_test = kpss(x=var_df, regression='c', nlags='auto', store=True)

C:\Users\dannydavis\AppData\Local\Temp\ipykernel_14164\838494665.py:1: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_test = kpss(x=var_df, regression='c', nlags='auto', store=True)


In [7]:
kpss_test

(np.float64(0.1598583779004237),
 np.float64(0.1),
 {'10%': 0.347, '5%': 0.463, '2.5%': 0.574, '1%': 0.739},
 <statsmodels.stats.diagnostic.ResultsStore at 0x12c261f28d0>)

In [12]:
kpss_resstore = kpss_test[-1]

In [25]:
stat, pval, crit_values, _ = kpss_test

print("=" * 45)
print("         KPSS Test Results")
print("=" * 45)
print(f"  KPSS Statistic  : {stat:.6f}")
print(f"  p-value         : {pval:.4f}")
print(f"  Lags used       : {kpss_resstore.lags}")
print(f"  Observations    : {kpss_resstore.nobs}")
print("-" * 45)
print("  Critical Values:")
for level, val in crit_values.items():
    marker = " <--" if stat < val else ""
    print(f"    {level:>4s} : {val:.3f}{marker}")
print("=" * 45)
reject = stat < crit_values['5%']
print(f"  H0 (stationary) : {'NOT rejected at 5%' if reject else 'Rejected at 5%'}")

         KPSS Test Results
  KPSS Statistic  : 0.159858
  p-value         : 0.1000
  Lags used       : 34
  Observations    : 4023
---------------------------------------------
  Critical Values:
     10% : 0.347 <--
      5% : 0.463 <--
    2.5% : 0.574 <--
      1% : 0.739 <--
  H0 (stationary) : NOT rejected at 5%
